This is an exploratory notebook for reading the data in this folder:
\\cryo-em\SURFACTANTE\RAW_DATA\TOMOS\Position_5\Processed

## Archivos de `Processed/` (Position_5)

Convenciones de IMOD. Deducido por los nombres, pendiente de confirmar con quien procesó los datos.

| Archivo | Qué es |
|---|---|
| `Position_5_MC.mrc` | Tilt series con corrección de movimiento (*MC*): un stack con una imagen por ángulo. |
| `Position_5_MC.rawtlt` | Ángulos de inclinación tal cual salieron del microscopio, uno por línea. |
| `Position_5_MC.tlt` | Ángulos finales, ya refinados tras el alineamiento. |
| `Position_5_MC_order_list.csv` | Orden en que se tomaron las imágenes (no se hicieron de menor a mayor ángulo). |
| `Position_5_MC.xf` | Transformaciones (desplazamiento, rotación) para alinear cada imagen con las demás. |
| `Position_5_MC.tltxf` | Las mismas transformaciones, ajustadas junto a los ángulos de inclinación. |
| `newst.com` | Receta de IMOD que aplica esas transformaciones y genera el stack alineado. |
| `Position_5_MC_ali.mrc` | Tilt series alineado, a resolución completa. |
| `Position_5_MC_bin4.mrc`, `_bin4_ali.mrc` | Las mismas imágenes reducidas 4×, sin alinear y alineadas. Sirven para trabajar rápido. |
| `Position_5_MC.defocus` | Desenfoque estimado por imagen (CTF), para corregir la distorsión del microscopio. |
| `tilt.com` | Receta de IMOD para reconstruir el tomograma. |
| `Position_5_SIRT_rec.mrc` | **Tomograma 3D**, reconstruido con SIRT. Es el único volumen real de la lista. |
| `Position_5_preview.mrc` | Vista previa pequeña (~2 MB), probablemente una reconstrucción reducida. |
| `reconstruction.json`, `tilt-series.json` | Metadatos del procesado y de la serie, escritos por la herramienta que lo hizo. |

**Flujo:** películas `.eer` (`Raw/`) → corrección de movimiento (`_MC`) → alineamiento con los ángulos (`.tlt`, `.xf`) → `_ali` → estimación de CTF (`.defocus`) → reconstrucción (`tilt.com`) → `SIRT_rec.mrc`.

**Ojo:** solo `SIRT_rec.mrc` es un volumen. En `_MC.mrc` y `_ali.mrc` el eje `z` es el ángulo de inclinación, no la profundidad.

# 1. Interpretación general de la carpeta

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mrcfile

BASE = Path(r"../datos/Processed")
POS = "Position_5"          # prefijo de los archivos

assert BASE.exists(), f"No existe {BASE}"
print("Carpeta:", BASE)

In [ ]:
#Inventariado de archivos:
rows = [(f.name, f.suffix, round(f.stat().st_size / 1e6, 2)) for f in sorted(BASE.iterdir()) if f.is_file()]
inv = pd.DataFrame(rows, columns=["archivo", "extensión", "MB"])
inv

In [ ]:
def read_numbers(path):
    return np.loadtxt(path)

def first_lines(path, n=15):
    with open(path, errors="replace") as fh:
        for i, line in enumerate(fh):
            if i >= n: print("..."); break
            print(line.rstrip())

# .rawtlt (microscopio) vs .tlt (tras alineamiento)
raw = read_numbers(BASE / f"{POS}_MC.rawtlt") if (BASE / f"{POS}_MC.rawtlt").exists() else None
tlt = read_numbers(BASE / f"{POS}_MC.tlt") if (BASE / f"{POS}_MC.tlt").exists() else None
print("rawtlt:", None if raw is None else (len(raw), raw.min(), raw.max()))
print("tlt   :", None if tlt is None else (len(tlt), tlt.min(), tlt.max()))

In [ ]:
order_file = BASE / f"{POS}_MC_order_list.csv"
if order_file.exists():
    order = pd.read_csv(order_file)
    display(order.head(10))
    print(order.shape)
    
order_file = BASE / f"{POS}_MC_order_list.csv"
if order_file.exists():
    order = pd.read_csv(order_file)
    display(order.head(10))
    print(order.shape)
    
if raw is not None:
    plt.figure(figsize=(6, 3))
    plt.plot(raw, "o-")
    plt.xlabel("nº de imagen (orden en el archivo)"); plt.ylabel("ángulo (°)")
    plt.title("Ángulos de inclinación"); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

Se aprecia que no ha habido descarte de imágenes y que se aumenta la inclinación de forma gradual

In [ ]:
# .xf: una fila por imagen, 6 números = matriz 2x2 + desplazamiento dx, dy
xf_file = BASE / f"{POS}_MC.xf"
if xf_file.exists():
    xf = read_numbers(xf_file)
    print("xf shape:", xf.shape, "| desplazamientos dx, dy en píxeles:")
    plt.figure(figsize=(6, 3)); plt.plot(xf[:, 4], label="dx"); plt.plot(xf[:, 5], label="dy")
    plt.legend(); plt.xlabel("imagen"); plt.ylabel("píxeles"); plt.tight_layout(); plt.show()

print("\n--- .defocus (primeras líneas) ---")
if (BASE / f"{POS}_MC.defocus").exists():
    first_lines(BASE / f"{POS}_MC.defocus", 8)

In [ ]:
for name in ["newst.com", "tilt.com"]:
    p = BASE / name
    if p.exists():
        print(f"\n===== {name} =====")
        first_lines(p, 40)

p = BASE / "tilt-series.json"
if p.exists():
    d = json.loads(p.read_text())
    print("\n===== tilt-series.json =====")
    print("campos:", list(d.keys()) if isinstance(d, dict) else type(d))
    print(json.dumps(d, indent=2)[:1500])

In [ ]:
p = BASE / "reconstruction.json"
d = json.loads(p.read_text())

print("elementos en la lista:", len(d))
print("tipo del primero:", type(d[0]))

if isinstance(d[0], dict):
    print("campos del primer elemento:", list(d[0].keys()))
    print(json.dumps(d[0], indent=2)[:2000])
else:
    print(d[0])

In [ ]:
info = []
for f in sorted(BASE.glob("*.mrc")):
    with mrcfile.mmap(f, permissive=True) as m:
        v = m.voxel_size
        info.append(dict(archivo=f.name, shape=tuple(m.data.shape), dtype=str(m.data.dtype),
                         voxel_A=round(float(v.x), 3), MB=round(f.stat().st_size / 1e6, 1)))
pd.DataFrame(info)

In [ ]:
TS = BASE / f"{POS}_MC_bin4_ali.mrc"        # bin4 = rápido; cambia a _ali.mrc para resolución completa
if not TS.exists():
    TS = BASE / f"{POS}_MC_bin4.mrc"
print("Usando:", TS.name)

ts = mrcfile.mmap(TS, permissive=True)
data = ts.data
print("shape:", data.shape, "| dtype:", data.dtype)

sample = np.asarray(data[:: max(1, data.shape[0] // 8)], dtype=np.float32)
vmin, vmax = np.percentile(sample, (1, 99))

angles = tlt if (tlt is not None and len(tlt) == data.shape[0]) else None

idx = np.linspace(0, data.shape[0] - 1, min(8, data.shape[0])).astype(int)
fig, axs = plt.subplots(2, 4, figsize=(14, 7)); axs = axs.ravel()
for a, i in zip(axs, idx):
    a.imshow(data[i], cmap="gray", vmin=vmin, vmax=vmax); a.axis("off")
    a.set_title(f"imagen {i}" + (f" ({angles[i]:.0f}°)" if angles is not None else ""))
plt.tight_layout(); plt.show()

In [ ]:
import ipywidgets as w

def show(i=0):
    plt.figure(figsize=(6, 6))
    plt.imshow(data[i], cmap="gray", vmin=vmin, vmax=vmax); plt.axis("off")
    plt.title(f"imagen {i}" + (f" - {angles[i]:.1f}°" if angles is not None else ""))
    plt.show()

w.interact(show, i=w.IntSlider(min=0, max=data.shape[0] - 1, step=1, value=0, continuous_update=False))

In [ ]:
REC = BASE / f"{POS}_SIRT_rec.mrc"
rec = mrcfile.mmap(REC, permissive=True)
vol = rec.data
print("shape:", vol.shape, "| vóxel (Å):", float(rec.voxel_size.x))

sub = np.asarray(vol[:: max(1, vol.shape[0] // 8), :: 4, :: 4], dtype=np.float32)
lo, hi = np.percentile(sub, (1, 99))
z, y, x = (s // 2 for s in vol.shape)

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(vol[z], cmap="gray", vmin=lo, vmax=hi); axs[0].set_title(f"corte eje 0 (índice {z})")
axs[1].imshow(vol[:, y, :], cmap="gray", vmin=lo, vmax=hi, aspect="auto"); axs[1].set_title(f"corte eje 1 (índice {y})")
axs[2].imshow(vol[:, :, x], cmap="gray", vmin=lo, vmax=hi, aspect="auto"); axs[2].set_title(f"corte eje 2 (índice {x})")
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6, 3)); plt.hist(sub.ravel(), bins=100, log=True)
plt.title("Histograma del tomograma (log)"); plt.tight_layout(); plt.show()

In [ ]:
from scipy import ndimage as ndi
from skimage.restoration import denoise_tv_chambolle

i = data.shape[0] // 2
img = np.asarray(data[i], dtype=np.float32)
img = (img - img.mean()) / img.std()

res = {
    "original": img,
    "gauss σ=1.5": ndi.gaussian_filter(img, 1.5),
    "mediana 3": ndi.median_filter(img, 3),
    "TV 0.2": denoise_tv_chambolle(img, weight=0.2),
}
fig, axs = plt.subplots(1, len(res), figsize=(4 * len(res), 4))
for a, (k, im) in zip(axs, res.items()):
    a.imshow(im, cmap="gray", vmin=-2.5, vmax=2.5); a.set_title(k); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# %gui qt        # descomenta si la ventana no aparece
import napari

viewer = napari.Viewer(ndisplay=3)

# Tomograma en 3D
lo, hi = np.percentile(np.asarray(vol[::4, ::4, ::4], dtype=np.float32), (1, 99))
viewer.add_image(np.asarray(vol), name="tomograma", contrast_limits=(lo, hi),
                 colormap="gray", rendering="attenuated_mip")

# Tilt series (cada corte = un ángulo), oculto al principio
viewer.add_image(np.asarray(data), name="tilt series (bin4 ali)", colormap="gray",
                 contrast_limits=(vmin, vmax), visible=False)

napari.run()

# 2. Comprobación con procesado de las imágenes

In [1]:
'''
En esta celda se importan las cosas necesarias y se definen las rutas
'''
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

%load_ext autoreload
%autoreload 2
%gui qt

import numpy as np
import napari
from scipy import ndimage as ndi
from functions import cargar_mrc, contraste

ruta = '../datos/Processed/Position_5_SIRT_rec.mrc'                     
tomo, vox = cargar_mrc(ruta)            
escala = (vox.z, vox.y, vox.x)                

In [3]:
'''
Comparativa de imágenes en 2D para ver efecto de diferentes filtros
'''
tomo = np.asarray(tomo, dtype=np.float32)
lo, hi = contraste(tomo)
versiones = {}
for s in (1, 2, 3):
    versiones[f"gauss σ={s}"] = ndi.gaussian_filter(tomo, sigma=3) 

# El multiplicador de n es un valor arbitrario, probar otros valores.  
n = np.clip((tomo - lo) / (hi - lo), 0, 1)
expo = np.exp(5 * n)

# k es un parámetro como el multiplicador de n, probar más valores
k = 10                                 
log = np.log1p(k * n) / np.log1p(k)  

                   
versiones = {"original": tomo, "gauss σ=1": versiones["gauss σ=1"], "gauss σ=2": versiones["gauss σ=2"], "gauss σ=3": versiones["gauss σ=3"] , "exponencial": expo, "logaritmico": log}
    
    
# 2. Visor en 2D
viewer = napari.Viewer(ndisplay=2)
for nombre, v in versiones.items():
    lo, hi = contraste(v)                          # ¿con qué función lo calculas? Una por versión
    viewer.add_image(v, name=nombre, contrast_limits=(lo, hi), colormap="gray", scale=escala)

# 3. Una al lado de otra, sincronizadas
viewer.canvas.grid.enabled = True

In [7]:
''' 
Aparentemente el que mejor funciona es el filtro gauss DT = 2. Se visualiza en 3D para poder hacer la comparativa
'''

paso = 2                                 # submuestreo: el 3D es pesado
sub_o = tomo[::paso, ::paso, ::paso]
sub_g = versiones["gauss σ=2"][::paso, ::paso, ::paso]
escala2 = tuple(s * paso for s in escala)    # cada vóxel ahora representa 'paso' veces más

invertir = False                         # pon True si las estructuras son más oscuras que el fondo
if invertir:
    sub_g = -sub_g

lo_o, hi_o = contraste(sub_o)
lo_g, hi_g = contraste(sub_g)
umbral = float(np.percentile(sub_g, 70))     # punto de partida; se ajusta en vivo en el panel

viewer3d = napari.Viewer(ndisplay=3)

viewer3d.add_image(sub_o, name="original", rendering="attenuated_mip",
                   colormap="gray", contrast_limits=(lo_o, hi_o),
                   opacity=0.3, scale=escala2)

viewer3d.add_image(sub_g, name="gauss σ=2 (iso)", rendering="iso", colormap = 'gray',
                   iso_threshold=umbral, 
                   contrast_limits=(lo_g, hi_g),
                   blending="translucent", scale=escala2)

<Image layer 'gauss σ=2 (iso)' at 0x222dbf1c190>